In [1]:
# Librerias

import re
import nltk

import pandas as pd
import numpy as np

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from nltk.corpus import stopwords

In [2]:
# Cargar datos

train = pd.read_csv('data/train.csv')
eval_df = pd.read_csv('data/eval.csv')

data = train.copy()

print("TAMAÑO DEL DATASET")
print(data.shape)
print(data.head())

print("\nDISTRIBUCIÓN DE LAS DECADAS")
print(data['decade'].value_counts())
print("\nNúmero de decadas:", data['decade'].nunique())

TAMAÑO DEL DATASET
(31403, 2)
                                                text  decade
0  \nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...     164
1  gone. Sus amigos , sus clientes, todo \ncuanto...     182
2  Prefosen quemanera,e per qualesfolpechas deuan...     157
3  Caistro  el  M  a  y  o  r  a  i  .]  Del  ape...     163
4  \nlos  que  panden  macho  ;  y \notros  en  l...     166

DISTRIBUCIÓN DE LAS DECADAS
decade
160    848
172    842
155    836
170    833
167    831
178    831
154    830
157    827
163    827
180    825
168    822
175    817
171    816
165    814
151    812
188    809
179    809
182    808
162    808
174    807
164    804
185    803
184    802
173    802
159    802
181    795
183    794
156    792
161    787
187    787
150    786
152    785
177    782
166    779
158    778
153    775
186    773
169    771
176    754
Name: count, dtype: int64

Número de decadas: 39


In [3]:
import re
import unicodedata
import nltk
from nltk.corpus import stopwords

# Configuración inicial
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""

    # 1. Normalización básica y limpieza de ruido OCR
    # Convertir a minúsculas y quitar saltos de línea/espacios múltiples
    texto = texto.lower()
    texto = re.sub(r'\n+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    
    # Quitar símbolos que claramente son ruido OCR (manteniendo letras y números)
    texto = re.sub(r'[^\w\sáéíóúüñ]', ' ', texto)

    # 2. Normalización de Variaciones Ortográficas (Siglos XVI-XVIII)
    # Estandarizar 'v' que actúa como 'u' al inicio de palabra (ej: 'vno' -> 'uno')
    texto = re.sub(r'\b[v]([aeiou])', r'u\1', texto)
    # Estandarizar la 's' larga (ſ) si existiera en la transcripción
    texto = texto.replace('ſ', 's')
    # Estandarizar 'x' antigua por 'j' en posiciones intervocálicas (ej: 'dixo' -> 'dijo')
    texto = re.sub(r'(\b\w+)x([aeiou])', r'\1j\2', texto)
    # Estandarizar 'y' final como 'i' (ej: 'muyo' -> 'muio')
    texto = re.sub(r'y\b', 'i', texto)

    # 3. Manejo de Acentos y Caracteres Especiales (NFKD)
    # Esto unifica "década" y "decada" en la misma forma base
    texto = unicodedata.normalize('NFKD', texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    # Re-normalizar a Unicode estándar tras quitar diacríticos
    texto = unicodedata.normalize('NFC', texto)

    # 4. Filtrado de palabras (Stopwords y longitud)
    palabras = texto.split()
    
    # Combinamos: Quitar stopwords + Quitar tokens de 1 o 2 caracteres (ruido OCR frecuente)
    # También eliminamos tokens que sean puramente numéricos
    palabras_limpias = [
        p for p in palabras 
        if p not in stop_words 
        and len(p) > 2 
        and not p.isdigit()
    ]

    return ' '.join(palabras_limpias)

# Aplicación del preprocesamiento
data['text_clean'] = data['text'].apply(limpiar_texto)
eval_df['text_clean'] = eval_df['text'].apply(limpiar_texto)

# Verificación de resultados
for decade in [150, 165, 180]:
    if not data[data['decade'] == decade].empty:
        ejemplo = data[data['decade'] == decade]['text_clean'].iloc[0]
        print(f"\n=== Década {decade} (Procesada) ===")
        print(ejemplo[:300])

[nltk_data] Downloading package stopwords to C:\Users\Juan
[nltk_data]     David\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



=== Década 150 (Procesada) ===
efiotnl fiit trce lleene leee ittc iieii iie oii ette ileiw temer tnner ellmpi

=== Década 165 (Procesada) ===
efto uina confejo portagal qual pbi precedido aragon nunca queri concurrir efl asproce fsione lamanos jun

=== Década 180 (Procesada) ===
obligacion ordenanzas imponen director general armada zelar mejoren cartas derroteros conformidad noticias deben darsele quanto descubrimientos nuevas tierras islas bajos sondas rectificacion acaso hiciese posiciones esten locadas cuidado duda alguna bueno pro uechoso capaz tambien producir armada r


In [4]:
# Ejemplo de léxico temporal (puedes ampliarlo según tu análisis)
lexico_temporal = {
    "vuestra_merced": ["merced", "vuestra", "habeis", "vmd"], # Siglos XVI-XVII
    "ilustrisimo": ["ilustre", "excelentisimo", "venerable"], # Siglos XVIII-XIX
    "internet": ["web", "digital", "computadora", "online"]   # Siglos XX-XXI
}

def aplicar_pesos_lexico(texto):
    palabras = texto.split()
    for i, palabra in enumerate(palabras):
        # Si la palabra es clave, la duplicamos para "engañar" al TF-IDF y darle más peso
        for categoria, terminos in lexico_temporal.items():
            if palabra in terminos:
                palabras[i] = (palabra + " ") * 2 # Duplica la importancia
    return ' '.join(palabras).strip()

# Aplicar antes de entrenar
data['text_clean'] = data['text_clean'].apply(aplicar_pesos_lexico)

In [5]:
from scipy.sparse import hstack

# Vectorizador de palabras
tfidf_word = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=200000,
    min_df=2,
    sublinear_tf=True,
)

# Vectorizador de caracteres
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=200000,
    min_df=2,
    sublinear_tf=True,
)

X_word = tfidf_word.fit_transform(data['text_clean'])
X_char = tfidf_char.fit_transform(data['text_clean'])
X_train = hstack([X_word, X_char])
y_train = data['decade']

print(f"Shape combinado: {X_train.shape}")

Shape combinado: (31403, 338122)


In [6]:
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import GridSearchCV

# 1. Definir los estimadores base
base_models = [
    ('nb', MultinomialNB()),
    ('lr', LogisticRegression(max_iter=2000))
]

# 2. Definir el meta-modelo (el que decide los pesos finales)
meta_model = LogisticRegression()

# 3. Crear el Stacking Classifier
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5 # Cross-validation interno para que el meta-modelo aprenda bien
)

# 4. Configurar el espacio de búsqueda para GridSearchCV
# Nota: usamos '__' para acceder a los parámetros de los modelos dentro del stacking
params = {
    'nb__alpha': [0.01, 0.1, 1.0],
    'lr__C': [0.1, 1, 10],
    'final_estimator__C': [0.1, 1.0]
}

# 5. Ejecutar la búsqueda (esto puede tardar unos minutos)
grid = GridSearchCV(stacking_model, params, cv=3, scoring='accuracy', n_jobs=-1)

# Suponiendo que ya tienes tu TF-IDF listo
grid.fit(X_train, y_train)

print(f"Mejor configuración: {grid.best_params_}")
print(f"Mejor Accuracy en entrenamiento: {grid.best_score_:.4f}")

KeyboardInterrupt: 